# EXP-17: Creative Pipeline — Hierarchical + CatBoost + SVD Topics + Self-Training

**Competition:** SPR 2026 — Mammography Report Classification  
**Task:** Classify mammography reports (Portuguese) into BI-RADS 0–6  
**Metric:** Macro F1  
**Runtime:** CPU only  

## Ideias criativas neste notebook:
1. **Classificacao Hierarquica** — primeiro separa Normal/Benigno/Suspeito/Maligno, depois refina dentro de cada grupo
2. **SVD/NMF Topic Features** — reducao de dimensionalidade da TF-IDF para capturar temas latentes
3. **CatBoost** — gradient boosting com suporte nativo a categoricas (nunca testado antes)
4. **Class Centroid Similarity** — similaridade cosseno do report a cada centroide de classe
5. **Self-Training (Pseudo-Labeling)** — retreinar modelos usando predicoes confiantes como dados extras
6. **Ordinal-Aware Post-Processing** — explora a natureza ordinal do BI-RADS (0<1<2<3<4<5<6)

In [ ]:
import os, re, hashlib, warnings, time, glob
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
    print('CatBoost available!')
except ImportError:
    !pip install -q catboost
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
    print('CatBoost installed and loaded.')

warnings.filterwarnings('ignore')
np.random.seed(42)

# --- Data Loading ---
def find_input(pattern):
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            if pattern in d.lower():
                return os.path.join(root, d)
    hits = glob.glob(f'/kaggle/input/**/*{pattern}*', recursive=True)
    return hits[0] if hits else None

COMP_DIR = find_input('spr-2026-mammography')
assert COMP_DIR, f'Competition data not found! Contents: {os.listdir("/kaggle/input")}'

train = pd.read_csv(os.path.join(COMP_DIR, 'train.csv'))
test  = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))

N_FOLDS = 5
N_CLASSES = 7

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train["target"].value_counts().sort_index()}')

In [ ]:
# =============================================================================
# Cell 2: Text Preprocessing + Section Extraction
# =============================================================================

def normalize_text(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    return s

def extract_achados(s):
    match = re.search(r'achados[:\s]*(.*?)(?:impress[aã]o|conclus[aã]o|an[aá]lise comparativa|opini[aã]o|$)', s, flags=re.DOTALL)
    return match.group(1).strip() if match else ""

def extract_conclusion(s):
    match = re.search(r'(?:impress[aã]o|conclus[aã]o|opini[aã]o)[:\s]*(.*?)(?:an[aá]lise comparativa|recomenda|$)', s, flags=re.DOTALL)
    return match.group(1).strip() if match else ""

def clean_text(s):
    s = normalize_text(s)
    s = re.sub(r'\b(cm|mm)\b', ' \\1 ', s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def stable_hash(s):
    return hashlib.md5(str(s).encode('utf-8')).hexdigest()

for df in [train, test]:
    df['normalized']  = df['report'].apply(normalize_text)
    df['achados']     = df['normalized'].apply(extract_achados)
    df['conclusion']  = df['normalized'].apply(extract_conclusion)
    df['clean_full']  = df['report'].apply(clean_text)
    df['clean_achados'] = df['achados'].apply(clean_text)

train['group'] = train['report'].apply(stable_hash)
y = train['target'].astype(int).values
groups = train['group'].values

print('Preprocessing done.')
print(f'Achados found: {(train["achados"] != "").sum()} / {len(train)}')

In [ ]:
# =============================================================================
# Cell 3: Dense Features + BI-RADS Explicit Extraction (enhanced)
# =============================================================================

def extract_birads_number(text):
    match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    return int(match.group(1)) if match else -1

def extract_dense_features(df):
    feat = pd.DataFrame(index=df.index)
    text = df['normalized']
    achados = df['achados']
    
    # Length / structure
    feat['report_length']     = text.str.len()
    feat['word_count']        = text.str.split().str.len().fillna(0).astype(int)
    feat['line_count']        = text.str.count('\n') + 1
    feat['achados_length']    = achados.str.len()
    feat['conclusion_length'] = df['conclusion'].str.len()
    feat['num_findings']      = achados.str.count(r'[\n\-\*\u2022]').fillna(0).astype(int)
    feat['achados_ratio']     = feat['achados_length'] / (feat['report_length'] + 1)
    feat['conclusion_ratio']  = feat['conclusion_length'] / (feat['report_length'] + 1)
    
    # Clinical keywords
    feat['has_measurement']    = text.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    feat['has_spiculation']    = text.str.contains(r'espiculad', regex=True).astype(int)
    feat['has_distortion']     = text.str.contains(r'distor[cç][aã]o arquitetural', regex=True).astype(int)
    feat['has_biopsy']         = text.str.contains(r'bi[oó]psia|carcinoma|resultado de cine', regex=True).astype(int)
    feat['has_nodule']         = text.str.contains(r'n[oó]dulo', regex=True).astype(int)
    feat['has_calcification']  = text.str.contains(r'calcifica[cç]', regex=True).astype(int)
    feat['has_asymmetry']      = text.str.contains(r'assimetria', regex=True).astype(int)
    feat['has_lymphnode']      = text.str.contains(r'linfonodo|axilar', regex=True).astype(int)
    feat['has_suspicious']     = text.str.contains(r'suspeit[oa]|malign[oa]', regex=True).astype(int)
    feat['has_benign']         = text.str.contains(r'benign[oa]|cisto[s]?\b', regex=True).astype(int)
    feat['has_gross_calc']     = text.str.contains(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', regex=True).astype(int)
    feat['has_nodulo_espic']   = text.str.contains(r'n[oó]dulo\s+espiculad', regex=True).astype(int)
    feat['has_heterogeneo']    = text.str.contains(r'heterog[eê]ne', regex=True).astype(int)
    feat['has_irregular']      = text.str.contains(r'irregular', regex=True).astype(int)
    feat['has_bilateral']      = text.str.contains(r'bilateral', regex=True).astype(int)
    
    # BI-RADS explicit
    feat['has_birads_explicit'] = text.str.contains(r'bi-?rads|categoria\s*\d', regex=True).astype(int)
    feat['birads_mentioned']    = text.apply(extract_birads_number)
    
    # --- NOVO: Composite risk score ---
    feat['risk_score'] = (
        feat['has_spiculation'] * 3 +
        feat['has_distortion'] * 3 +
        feat['has_nodulo_espic'] * 4 +
        feat['has_biopsy'] * 5 +
        feat['has_suspicious'] * 3 +
        feat['has_irregular'] * 2 +
        feat['has_heterogeneo'] * 1 +
        feat['has_calcification'] * 1 +
        feat['has_asymmetry'] * 2 -
        feat['has_benign'] * 2 -
        feat['has_gross_calc'] * 2
    )
    
    return feat

dense_train = extract_dense_features(train)
dense_test  = extract_dense_features(test)
X_train_dense = csr_matrix(dense_train.values.astype(np.float32))
X_test_dense  = csr_matrix(dense_test.values.astype(np.float32))

print(f'Dense features: {X_train_dense.shape[1]}')
print(f'Risk score range: [{dense_train["risk_score"].min()}, {dense_train["risk_score"].max()}]')

In [ ]:
# =============================================================================
# Cell 4: TF-IDF + SVD/NMF Topic Features
# =============================================================================

print('Building TF-IDF features...')

# Standard TF-IDF (full text)
tfidf_full_word = TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)
tfidf_full_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 6), min_df=3, max_df=0.95,
                                   sublinear_tf=True, max_features=100000)

X_tr_word = tfidf_full_word.fit_transform(train['clean_full'])
X_te_word = tfidf_full_word.transform(test['clean_full'])
X_tr_char = tfidf_full_char.fit_transform(train['clean_full'])
X_te_char = tfidf_full_char.transform(test['clean_full'])

# Achados TF-IDF
tfidf_ach = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_tr_ach = tfidf_ach.fit_transform(train['clean_achados'])
X_te_ach = tfidf_ach.transform(test['clean_achados'])

# Sparse combined
X_train_sparse = hstack([X_tr_word, X_tr_char, X_tr_ach]).tocsr()
X_test_sparse  = hstack([X_te_word, X_te_char, X_te_ach]).tocsr()

print(f'Sparse TF-IDF features: {X_train_sparse.shape[1]}')

# --- CRIATIVO: SVD Topic Features ---
print('\nComputing SVD topic features (50 components)...')
svd = TruncatedSVD(n_components=50, random_state=42)
X_tr_svd = svd.fit_transform(X_train_sparse)
X_te_svd = svd.transform(X_test_sparse)
print(f'SVD explained variance: {svd.explained_variance_ratio_.sum():.3f}')

# --- CRIATIVO: NMF Topic Features ---
print('Computing NMF topic features (30 components)...')
# NMF precisa de input nao-negativo — TF-IDF ja e
nmf = NMF(n_components=30, random_state=42, max_iter=300)
X_tr_nmf = nmf.fit_transform(X_train_sparse)
X_te_nmf = nmf.transform(X_test_sparse)
print(f'NMF reconstruction error: {nmf.reconstruction_err_:.1f}')

# Full feature matrix (sparse + dense + SVD + NMF)
X_train_full = hstack([
    X_train_sparse, X_train_dense,
    csr_matrix(X_tr_svd), csr_matrix(X_tr_nmf)
]).tocsr()
X_test_full = hstack([
    X_test_sparse, X_test_dense,
    csr_matrix(X_te_svd), csr_matrix(X_te_nmf)
]).tocsr()

print(f'\nTotal features: {X_train_full.shape[1]}')

In [ ]:
# =============================================================================
# Cell 5: CRIATIVO — Class Centroid Similarity Features
# Calcula a similaridade cosseno de cada report ao centroide de cada classe
# Isso da ao modelo uma nocao de "distancia" ao prototipo de cada BI-RADS
# =============================================================================

print('Computing class centroid similarity features...')

# Usar SVD features (dense, 50 dims) para calcular centroides — mais estavel que sparse
centroids = np.zeros((N_CLASSES, X_tr_svd.shape[1]))
for cls in range(N_CLASSES):
    mask = y == cls
    if mask.sum() > 0:
        centroids[cls] = X_tr_svd[mask].mean(axis=0)

# Similaridade cosseno de cada sample a cada centroide
X_tr_centroid_sim = cosine_similarity(X_tr_svd, centroids)  # (n_train, 7)
X_te_centroid_sim = cosine_similarity(X_te_svd, centroids)  # (n_test, 7)

# Features extras: max_sim, argmax_sim, spread (max - 2nd max)
def centroid_meta_features(sim_matrix):
    feat = pd.DataFrame()
    feat['max_centroid_sim'] = sim_matrix.max(axis=1)
    feat['argmax_centroid']  = sim_matrix.argmax(axis=1)
    sorted_sims = np.sort(sim_matrix, axis=1)[:, ::-1]
    feat['centroid_spread'] = sorted_sims[:, 0] - sorted_sims[:, 1]  # confidence
    feat['centroid_entropy'] = -(sim_matrix * np.log(np.clip(sim_matrix, 1e-10, 1))).sum(axis=1)
    return feat

centroid_meta_tr = centroid_meta_features(X_tr_centroid_sim)
centroid_meta_te = centroid_meta_features(X_te_centroid_sim)

# Combinar: 7 sims + 4 meta features = 11 features de centroide
X_tr_centroid_all = np.hstack([X_tr_centroid_sim, centroid_meta_tr.values])
X_te_centroid_all = np.hstack([X_te_centroid_sim, centroid_meta_te.values])

print(f'Centroid features: {X_tr_centroid_all.shape[1]}')
print(f'Centroid argmax accuracy: {(X_tr_centroid_sim.argmax(axis=1) == y).mean():.3f}')

In [ ]:
# =============================================================================
# Cell 6: CRIATIVO — Hierarchical Classification
# Stage 1: Classifica em 4 macro-categorias (Normal, Benigno, Suspeito, Maligno)
# Stage 2: Refina dentro de cada macro-categoria para BI-RADS exato
# =============================================================================

print('='*60)
print('HIERARCHICAL CLASSIFICATION')
print('='*60)

# Mapeamento hierarquico baseado em conhecimento clinico:
# Macro-grupo 0: Incompleto/Normal → BI-RADS 0, 1
# Macro-grupo 1: Benigno           → BI-RADS 2
# Macro-grupo 2: Provavelmente benigno / Suspeito baixo → BI-RADS 3
# Macro-grupo 3: Suspeito alto / Maligno → BI-RADS 4, 5, 6

MACRO_MAP = {0: 0, 1: 0, 2: 1, 3: 2, 4: 3, 5: 3, 6: 3}
y_macro = np.array([MACRO_MAP[v] for v in y])

print(f'Macro-class distribution: {pd.Series(y_macro).value_counts().sort_index().to_dict()}')

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_sparse, y, groups))

# Stage 1: Macro-classifier
oof_macro = np.zeros((len(train), 4), dtype=np.float64)
te_macro  = np.zeros((len(test), 4), dtype=np.float64)

for fi, (tr_idx, va_idx) in enumerate(folds):
    m = CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', C=1.0, random_state=42, max_iter=15000),
        cv=3, method='sigmoid'
    )
    m.fit(X_train_sparse[tr_idx], y_macro[tr_idx])
    oof_macro[va_idx] = m.predict_proba(X_train_sparse[va_idx])
    te_macro += m.predict_proba(X_test_sparse) / N_FOLDS
    fold_f1 = f1_score(y_macro[va_idx], np.argmax(oof_macro[va_idx], axis=1), average='macro')
    print(f'  Stage 1 Fold {fi}: macro-F1 = {fold_f1:.4f}')

macro_preds = np.argmax(oof_macro, axis=1)
print(f'Stage 1 OOF macro-F1 (4 classes): {f1_score(y_macro, macro_preds, average="macro"):.4f}')

# Stage 2: Sub-classifiers para grupos que tem >1 classe
# Grupo 0 (BI-RADS 0,1): precisa separar 0 vs 1
# Grupo 3 (BI-RADS 4,5,6): precisa separar 4 vs 5 vs 6
# Grupos 1 e 2 tem apenas 1 classe cada → nao precisa stage 2

oof_hierarchical = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_hierarchical  = np.zeros((len(test), N_CLASSES), dtype=np.float64)

# Sub-classifier: Grupo 0 (BI-RADS 0 vs 1)
print('\n--- Stage 2: Separating BI-RADS 0 vs 1 ---')
mask_01_tr = np.isin(y, [0, 1])
if mask_01_tr.sum() > 50:
    oof_sub01 = np.zeros((len(train), 2), dtype=np.float64)
    te_sub01  = np.zeros((len(test), 2), dtype=np.float64)
    y_sub01 = y[mask_01_tr]  # original labels (0 or 1)
    # Remap to 0,1 for this sub-problem
    label_map_01 = {0: 0, 1: 1}
    y_sub01_mapped = np.array([label_map_01[v] for v in y_sub01])
    
    for fi, (tr_idx, va_idx) in enumerate(folds):
        # Filter to only macro-group 0 samples in this fold
        tr_sub = tr_idx[np.isin(y[tr_idx], [0, 1])]
        va_sub = va_idx[np.isin(y[va_idx], [0, 1])]
        if len(tr_sub) < 10 or len(va_sub) < 5:
            continue
        m = CalibratedClassifierCV(
            LinearSVC(class_weight='balanced', C=1.0, random_state=42, max_iter=15000),
            cv=3, method='sigmoid'
        )
        y_tr_sub = np.array([label_map_01[v] for v in y[tr_sub]])
        m.fit(X_train_full[tr_sub], y_tr_sub)
        oof_sub01[va_sub] = m.predict_proba(X_train_full[va_sub])
        te_sub01 += m.predict_proba(X_test_full) / N_FOLDS
    print(f'  Trained sub-classifier for BI-RADS 0 vs 1')

# Sub-classifier: Grupo 3 (BI-RADS 4 vs 5 vs 6)
print('--- Stage 2: Separating BI-RADS 4 vs 5 vs 6 ---')
mask_456_tr = np.isin(y, [4, 5, 6])
if mask_456_tr.sum() > 20:
    oof_sub456 = np.zeros((len(train), 3), dtype=np.float64)
    te_sub456  = np.zeros((len(test), 3), dtype=np.float64)
    label_map_456 = {4: 0, 5: 1, 6: 2}
    
    for fi, (tr_idx, va_idx) in enumerate(folds):
        tr_sub = tr_idx[np.isin(y[tr_idx], [4, 5, 6])]
        va_sub = va_idx[np.isin(y[va_idx], [4, 5, 6])]
        if len(tr_sub) < 10 or len(va_sub) < 3:
            continue
        m = CalibratedClassifierCV(
            LinearSVC(class_weight='balanced', C=0.5, random_state=42, max_iter=15000),
            cv=2, method='sigmoid'
        )
        y_tr_sub = np.array([label_map_456[v] for v in y[tr_sub]])
        m.fit(X_train_full[tr_sub], y_tr_sub)
        oof_sub456[va_sub] = m.predict_proba(X_train_full[va_sub])
        te_sub456 += m.predict_proba(X_test_full) / N_FOLDS
    print(f'  Trained sub-classifier for BI-RADS 4 vs 5 vs 6')

# Combinar hierarchical predictions:
# P(BI-RADS=k) = P(macro_group) * P(sub_class | macro_group)
for i in range(len(train)):
    # Grupo 0 → BI-RADS 0, 1
    oof_hierarchical[i, 0] = oof_macro[i, 0] * oof_sub01[i, 0]
    oof_hierarchical[i, 1] = oof_macro[i, 0] * oof_sub01[i, 1]
    # Grupo 1 → BI-RADS 2
    oof_hierarchical[i, 2] = oof_macro[i, 1]
    # Grupo 2 → BI-RADS 3
    oof_hierarchical[i, 3] = oof_macro[i, 2]
    # Grupo 3 → BI-RADS 4, 5, 6
    oof_hierarchical[i, 4] = oof_macro[i, 3] * oof_sub456[i, 0]
    oof_hierarchical[i, 5] = oof_macro[i, 3] * oof_sub456[i, 1]
    oof_hierarchical[i, 6] = oof_macro[i, 3] * oof_sub456[i, 2]

for i in range(len(test)):
    te_hierarchical[i, 0] = te_macro[i, 0] * te_sub01[i, 0]
    te_hierarchical[i, 1] = te_macro[i, 0] * te_sub01[i, 1]
    te_hierarchical[i, 2] = te_macro[i, 1]
    te_hierarchical[i, 3] = te_macro[i, 2]
    te_hierarchical[i, 4] = te_macro[i, 3] * te_sub456[i, 0]
    te_hierarchical[i, 5] = te_macro[i, 3] * te_sub456[i, 1]
    te_hierarchical[i, 6] = te_macro[i, 3] * te_sub456[i, 2]

hier_f1 = f1_score(y, np.argmax(oof_hierarchical, axis=1), average='macro')
print(f'\nHierarchical OOF macro-F1: {hier_f1:.4f}')

In [ ]:
# =============================================================================
# Cell 7: Flat Models — SVC + LGB + CatBoost (com features enriquecidas)
# =============================================================================

print('='*60)
print('FLAT MODELS (with enriched features)')
print('='*60)

# Features para LGB/CatBoost: sparse + dense + SVD + NMF + centroid sim
X_tr_enriched = hstack([
    X_train_sparse, X_train_dense,
    csr_matrix(X_tr_svd), csr_matrix(X_tr_nmf),
    csr_matrix(X_tr_centroid_all)
]).tocsr()
X_te_enriched = hstack([
    X_test_sparse, X_test_dense,
    csr_matrix(X_te_svd), csr_matrix(X_te_nmf),
    csr_matrix(X_te_centroid_all)
]).tocsr()

print(f'Enriched features: {X_tr_enriched.shape[1]}')

oof_flat = {}  # model_name -> (n, 7)
te_flat  = {}

# --- SVC on sparse ---
print('\nTraining SVC...')
oof_svc = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_svc  = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', C=1.0, random_state=42, max_iter=15000),
        cv=3, method='sigmoid'
    )
    m.fit(X_train_sparse[tr_idx], y[tr_idx])
    oof_svc[va_idx] = m.predict_proba(X_train_sparse[va_idx])
    te_svc += m.predict_proba(X_test_sparse) / N_FOLDS
svc_f1 = f1_score(y, np.argmax(oof_svc, axis=1), average='macro')
print(f'  SVC OOF F1: {svc_f1:.4f}')
oof_flat['svc'] = oof_svc; te_flat['svc'] = te_svc

# --- LightGBM on enriched ---
print('Training LightGBM...')
oof_lgb = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_lgb  = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = lgb.LGBMClassifier(class_weight='balanced', n_estimators=500, learning_rate=0.03,
                            max_depth=7, num_leaves=63, subsample=0.8, colsample_bytree=0.6,
                            random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_tr_enriched[tr_idx], y[tr_idx])
    oof_lgb[va_idx] = m.predict_proba(X_tr_enriched[va_idx])
    te_lgb += m.predict_proba(X_te_enriched) / N_FOLDS
lgb_f1 = f1_score(y, np.argmax(oof_lgb, axis=1), average='macro')
print(f'  LGB OOF F1: {lgb_f1:.4f}')
oof_flat['lgb'] = oof_lgb; te_flat['lgb'] = te_lgb

# --- CatBoost on enriched (NOVO!) ---
print('Training CatBoost...')
oof_cb = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_cb  = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    # CatBoost precisa de array denso para sparse (ou Pool)
    m = CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6, l2_leaf_reg=3,
        auto_class_weights='Balanced', random_seed=42, verbose=0,
        task_type='CPU'
    )
    # Usar SVD + NMF + dense + centroid (features densas ~100 dims)
    X_cb_tr = np.hstack([X_tr_svd[tr_idx], X_tr_nmf[tr_idx],
                         dense_train.values[tr_idx], X_tr_centroid_all[tr_idx]])
    X_cb_va = np.hstack([X_tr_svd[va_idx], X_tr_nmf[va_idx],
                         dense_train.values[va_idx], X_tr_centroid_all[va_idx]])
    X_cb_te = np.hstack([X_te_svd, X_te_nmf, dense_test.values, X_te_centroid_all])
    
    m.fit(X_cb_tr, y[tr_idx], eval_set=(X_cb_va, y[va_idx]), early_stopping_rounds=50)
    oof_cb[va_idx] = m.predict_proba(X_cb_va)
    te_cb += m.predict_proba(X_cb_te) / N_FOLDS
cb_f1 = f1_score(y, np.argmax(oof_cb, axis=1), average='macro')
print(f'  CatBoost OOF F1: {cb_f1:.4f}')
oof_flat['catboost'] = oof_cb; te_flat['catboost'] = te_cb

# --- LogReg on enriched ---
print('Training LogisticRegression...')
oof_lr = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_lr  = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = LogisticRegression(class_weight='balanced', C=1.0, max_iter=5000,
                           solver='lbfgs', multi_class='multinomial', random_state=42, n_jobs=-1)
    m.fit(X_tr_enriched[tr_idx], y[tr_idx])
    oof_lr[va_idx] = m.predict_proba(X_tr_enriched[va_idx])
    te_lr += m.predict_proba(X_te_enriched) / N_FOLDS
lr_f1 = f1_score(y, np.argmax(oof_lr, axis=1), average='macro')
print(f'  LogReg OOF F1: {lr_f1:.4f}')
oof_flat['logreg'] = oof_lr; te_flat['logreg'] = te_lr

In [ ]:
# =============================================================================
# Cell 8: CRIATIVO — Self-Training (Pseudo-Labeling)
# Usa predicoes confiantes do ensemble como dados extras de treino
# =============================================================================

print('='*60)
print('SELF-TRAINING (Pseudo-Labeling)')
print('='*60)

# Ensemble basico para selecionar pseudo-labels
oof_base_ensemble = 0.35 * oof_svc + 0.25 * oof_lgb + 0.20 * oof_cb + 0.20 * oof_lr
base_f1 = f1_score(y, np.argmax(oof_base_ensemble, axis=1), average='macro')
print(f'Base ensemble OOF F1: {base_f1:.4f}')

# Identificar samples de alta confianca (>0.95 prob) para classes raras
# Esses podem ser usados como pseudo-labels para retreinar modelos
confidence_threshold = 0.90
max_prob = oof_base_ensemble.max(axis=1)
pred_class = oof_base_ensemble.argmax(axis=1)
correct_confident = (max_prob > confidence_threshold) & (pred_class == y)

print(f'\nHigh-confidence predictions (>{confidence_threshold}):')
print(f'  Total: {(max_prob > confidence_threshold).sum()} / {len(y)}')
print(f'  Correct: {correct_confident.sum()}')
for cls in range(N_CLASSES):
    mask = (max_prob > confidence_threshold) & (pred_class == cls)
    total_cls = (y == cls).sum()
    print(f'  Class {cls}: {mask.sum()} confident ({(mask & (y==cls)).sum()} correct) / {total_cls} total')

# Self-training: retreinar SVC com sample weighting
# Dar peso extra para samples com alta confianca nas classes raras
print('\nSelf-training SVC with confidence-weighted samples...')
sample_weights = np.ones(len(y))
for cls in [3, 4, 5, 6]:  # classes raras
    confident_mask = (max_prob > confidence_threshold) & (pred_class == cls) & (y == cls)
    sample_weights[confident_mask] *= 2.0  # dobra o peso de exemplos confiantes raros

oof_svc_st = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_svc_st  = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', C=1.0, random_state=42, max_iter=15000),
        cv=3, method='sigmoid'
    )
    m.fit(X_train_sparse[tr_idx], y[tr_idx], sample_weight=sample_weights[tr_idx])
    oof_svc_st[va_idx] = m.predict_proba(X_train_sparse[va_idx])
    te_svc_st += m.predict_proba(X_test_sparse) / N_FOLDS

st_f1 = f1_score(y, np.argmax(oof_svc_st, axis=1), average='macro')
print(f'Self-trained SVC OOF F1: {st_f1:.4f} (vs original SVC: {svc_f1:.4f})')
oof_flat['svc_st'] = oof_svc_st; te_flat['svc_st'] = te_svc_st

In [ ]:
# =============================================================================
# Cell 9: Mega Ensemble — Flat + Hierarchical + Self-Trained
# =============================================================================

print('='*60)
print('MEGA ENSEMBLE: Flat + Hierarchical + Self-Trained')
print('='*60)

# Todos os modelos disponiveis:
all_oof = {
    'svc': oof_svc,
    'lgb': oof_lgb,
    'catboost': oof_cb,
    'logreg': oof_lr,
    'svc_st': oof_svc_st,
    'hierarchical': oof_hierarchical,
}
all_test = {
    'svc': te_svc,
    'lgb': te_lgb,
    'catboost': te_cb,
    'logreg': te_lr,
    'svc_st': te_svc_st,
    'hierarchical': te_hierarchical,
}

model_names = list(all_oof.keys())
n_models = len(model_names)

# Print individual scores
for name in model_names:
    f1 = f1_score(y, np.argmax(all_oof[name], axis=1), average='macro')
    print(f'  {name}: OOF F1 = {f1:.4f}')

# Dirichlet weight search
print('\nSearching optimal weights (20k combinations)...')
best_f1 = 0
best_weights = None
np.random.seed(42)

for _ in range(20000):
    raw = np.random.dirichlet(np.ones(n_models))
    w = np.round(raw / 0.05) * 0.05
    if w.sum() == 0: continue
    w = w / w.sum()
    blend = sum(w[i] * all_oof[model_names[i]] for i in range(n_models))
    score = f1_score(y, np.argmax(blend, axis=1), average='macro')
    if score > best_f1:
        best_f1 = score
        best_weights = w.copy()

print(f'\nBest weights (OOF F1={best_f1:.4f}):')
for name, w in zip(model_names, best_weights):
    print(f'  {name}: {w:.2f}')

# Build final blend
oof_blend = sum(best_weights[i] * all_oof[model_names[i]] for i in range(n_models))
te_blend  = sum(best_weights[i] * all_test[model_names[i]] for i in range(n_models))

In [ ]:
# =============================================================================
# Cell 10: Threshold Search + Ordinal Post-Processing
# =============================================================================

threshold_classes = [6, 5, 4, 3, 1, 0]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    preds = np.argmax(proba, axis=1).copy()
    for cls in threshold_classes:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for higher_cls in threshold_classes:
                if higher_cls == cls: break
                if higher_cls in thresholds:
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

baseline_preds = np.argmax(oof_blend, axis=1)
baseline_f1 = f1_score(y, baseline_preds, average='macro')
print(f'Baseline OOF macro-F1 (no thresholds): {baseline_f1:.4f}')

best_thresholds = {}
current_f1 = baseline_f1
for cls in threshold_classes:
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_blend, trial)
        trial_f1 = f1_score(y, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f'Class {cls}: threshold={best_cls_thr:.2f}, macro-F1={best_cls_f1:.4f}')
    else:
        print(f'Class {cls}: no improvement')

final_oof_preds = apply_thresholds(oof_blend, best_thresholds)
final_oof_f1 = f1_score(y, final_oof_preds, average='macro')
print(f'\nFinal OOF macro-F1 with thresholds: {final_oof_f1:.4f}')
print(f'Thresholds: {best_thresholds}')

# --- CRIATIVO: Ordinal-Aware Post-Processing ---
# Se um report tem proba alta em 4 E 5, mas o modelo diz 3, isso e suspeito
# Verifica consistencia ordinal: P(>=k) deve ser monotonicamente decrescente
print('\nOrdinal consistency check...')
ordinal_fixes = 0
oof_ordinal_preds = final_oof_preds.copy()
for i in range(len(oof_ordinal_preds)):
    pred = oof_ordinal_preds[i]
    proba = oof_blend[i]
    # Se proba cumulativa P(>=4) > 0.3 mas modelo diz 2, upgrade para 3
    p_high = proba[4:].sum()
    if p_high > 0.30 and pred == 2:
        oof_ordinal_preds[i] = 3
        ordinal_fixes += 1
    # Se proba cumulativa P(>=5) > 0.3 mas modelo diz <4, upgrade
    elif proba[5:].sum() > 0.25 and pred < 4:
        oof_ordinal_preds[i] = 4
        ordinal_fixes += 1

ordinal_f1 = f1_score(y, oof_ordinal_preds, average='macro')
print(f'Ordinal fixes: {ordinal_fixes}')
print(f'After ordinal correction: {ordinal_f1:.4f} (vs {final_oof_f1:.4f})')

# Usar ordinal se melhorou, senao manter original
USE_ORDINAL = ordinal_f1 > final_oof_f1
print(f'Using ordinal correction: {USE_ORDINAL}')

In [ ]:
# =============================================================================
# Cell 11: Clinical Guardrails + Final Submission
# =============================================================================

test_preds = apply_thresholds(te_blend, best_thresholds)

# Apply ordinal correction if it helped on OOF
if USE_ORDINAL:
    for i in range(len(test_preds)):
        pred = test_preds[i]
        proba = te_blend[i]
        p_high = proba[4:].sum()
        if p_high > 0.30 and pred == 2:
            test_preds[i] = 3
        elif proba[5:].sum() > 0.25 and pred < 4:
            test_preds[i] = 4

test['target'] = test_preds

def apply_clinical_guardrails(row):
    text = str(row['report']).lower()
    pred = int(row['target'])
    
    # Explicit BI-RADS mention
    birads_match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    if birads_match:
        mentioned = int(birads_match.group(1))
        if 0 <= mentioned <= 6:
            return mentioned
    
    # Confirmed malignancy
    if re.search(r'resultado de cine grau 3|carcinoma|neoplasia maligna', text):
        return 6
    
    # Highly suspicious morphology
    if re.search(r'n[oó]dulo\s+espiculad', text) and pred < 4:
        return 5
    if 'espiculad' in text and 'distor' in text and pred < 4:
        return 5
    
    # Benign calc downgrade
    if re.search(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', text):
        if pred >= 4 and not re.search(r'espiculad|suspeit|malign|distor', text):
            return 2
    
    return pred

test['target'] = test.apply(apply_clinical_guardrails, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('Submission saved!')
print(f'\nPrediction distribution:\n{submission["target"].value_counts().sort_index()}')
print(f'\n{submission}')